## Vaccine refusal

Pull all visits with a vaccine refusal code from the Observations table
Save into '/share/pi/deho-pi/AFC/mortonc/intermediate/vax_refusal.csv'

(NOTE: import pandas as pd
import numpy as np
import os
import zipfileOriginally, this file was located in measles_codes_dates_data)

"Marker 2: Visit diagnosis. This visit-level marker relies on vaccine refusal diagnosis
codes derived from the International Classification of Diseases (ICD) diagnosis codes
(ICD-9 codes: V64.00, V64.05, V64.09, V64.06 or V64.07 and ICD-10 codes: Z28.1,
Z28.2x, Z28.82, Z28.83, Z28.89, Z28.9)."

In [ ]:
import pandas as pd
import numpy as np
import os
import zipfile

In [ ]:
def save_zip_csv(filepath, dataset):
    # write to CSV
    csv_filename = filepath
    dataset.to_csv(csv_filename, index=False)

    # zip CSV
    zip_filename = csv_filename + '.zip'

    with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
        zipf.write(csv_filename, os.path.basename(csv_filename))

    # remove large csv
    os.remove(csv_filename)
    
    print("Saved!")

In [ ]:
# function that determines what kind of file we're dealing with and reads it into a pd DataFrame appropriately
def read_multi(path):
    # .csv
    if path.endswith('.csv'):
        data = pd.read_csv(path)
        return data
    
    # .csv.gz
    elif path.endswith('.csv.gz'):
        data = pd.read_csv(path, compression = 'gzip')
        return data
    
    elif path.endswith('.csv.zip'):
        data = pd.read_csv(path, compression='gzip', low_memory = False)
        return data
    
    # .pkl
    elif path.endswith('.pkl'):
        data = pd.read_pickle(path)
        return data
    
    else:
        print(path)

In [ ]:
# person id <-> patientuid
person = pd.read_csv('/share/pi/deho-pi/AFC/BQ/person_0524.csv.gz')
person = person[['person_id', 'person_source_value']]

In [ ]:
path = "/share/pi/deho/AFC/BQ/observation_0724/"
path_list = os.listdir(path)
obs_paths = pd.Series(path_list)[pd.Series(path_list).str.startswith("obs")]

In [ ]:
refusal_source_values = ['V64.00', 'V64.05', 'V64.09', 'V64.06','V64.07','Z28.1', 
                         'Z28.20', 'Z28.21', 'Z28.22', 'Z28.23', 'Z28.24', 'Z28.25',
                         'Z28.26', 'Z28.27', 'Z28.28', 'Z28.29', 
                         'Z28.82', 'Z28.83', 'Z28.89', 'Z28.9']

In [ ]:
obs = pd.DataFrame(['Unnamed: 0', 'observation_id', 'person_id', 'observation_concept_id',
       'observation_source_value', 'observation_date',
       'observation_type_concept_id', 'value_as_string', 'value_as_concept_id',
       'unit_concept_id', 'visit_occurrence_id', 'observation_event_id'])

for obs_name in obs_paths:
    obs_path = path + obs_name
    obs_data = read_multi(obs_path)
    
    obs_data = obs_data[obs_data['observation_source_value'].isin(refusal_source_values)]
    obs = pd.concat([obs, obs_data])
    
    print(obs_name)

In [ ]:
len(obs)

In [ ]:
len(obs['person_id'].unique())

In [ ]:
obs = pd.merge(obs, person)

In [ ]:
len(obs['person_source_value'].unique())

In [ ]:
obs = obs.rename(columns={"person_source_value": "patientuid"}, errors="raise")

In [ ]:
save_zip_csv('/share/pi/deho-pi/AFC/mortonc/intermediate/vax_refusal.csv', obs)